# CampusIQ — Data Preprocessing & Exploratory Data Analysis (EDA)

**Project:** Intelligent Campus Crowd Prediction & Analytics System  
**Course:** Statistical Machine Learning  
**Dataset:** Synthetic campus occupancy data (`campus_crowd.csv`)

## Objective
This notebook loads the synthetic dataset, checks its quality, performs basic preprocessing, creates useful time-based features, and explores occupancy patterns for the **Library** and **Cafeteria**.

> **Important:** The dataset is synthetic and intended for academic demonstration. Results do not represent live measurements from a real university campus.

### Expected dataset columns
`date`, `day_of_week`, `hour`, `is_weekend`, `exam_period`, `campus_event`, `location`, `capacity`, `occupancy_count`, `occupancy_pct`


## 1. Import required libraries

We use:
- **Pandas** for loading and manipulating tabular data.
- **NumPy** for numerical operations.
- **Matplotlib** for visualizations.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# The notebook expects to be run from the CampusIQ project folder.
DATA_PATH = Path("data") / "campus_crowd.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Open this notebook from the CampusIQ project folder "
        "and confirm that data/campus_crowd.csv exists."
    )

print("Libraries imported and dataset path verified.")

## 2. Load the dataset

We parse the `date` column as a date type so that it can be used for time-based analysis.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["date"])

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())

## 3. Understand the dataset structure

First, inspect column names, data types, and summary information. This helps identify unexpected types or missing values.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all").T)

## 4. Data quality checks

We check for missing values, duplicate records, invalid occupancy counts, and occupancy percentages outside the valid 0–100% range.

In [ ]:
print("Missing values per column:")
display(df.isna().sum().to_frame("missing_count"))

print(f"\nDuplicate rows: {df.duplicated().sum():,}")

# Domain validation checks
invalid_count = ((df["occupancy_count"] < 0) | (df["occupancy_count"] > df["capacity"])).sum()
invalid_pct = ((df["occupancy_pct"] < 0) | (df["occupancy_pct"] > 100)).sum()

print(f"Rows with invalid occupancy_count: {invalid_count}")
print(f"Rows with invalid occupancy_pct: {invalid_pct}")

print("\nUnique locations:")
print(df["location"].unique())

## 5. Preprocessing

For this synthetic dataset:
1. Remove exact duplicate rows, if any.
2. Convert categorical and numeric columns to appropriate types.
3. Recalculate occupancy percentage from count and capacity to ensure consistency.
4. Create readable day names and cyclical time features for later modelling.

The cyclical hour features represent the fact that hours close to midnight are close to each other in time. The generated dataset currently covers 08:00–22:00, but this encoding is still useful for the modelling stage.

In [ ]:
# Work on a copy so the raw DataFrame remains available for reference.
data = df.copy()

# Remove exact duplicate records.
data = data.drop_duplicates().reset_index(drop=True)

# Ensure correct data types.
data["date"] = pd.to_datetime(data["date"], errors="coerce")
numeric_columns = [
    "day_of_week", "hour", "is_weekend", "exam_period",
    "campus_event", "capacity", "occupancy_count", "occupancy_pct"
]
for col in numeric_columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

data["location"] = data["location"].astype("category")

# Remove rows with missing essential fields.
required_columns = ["date", "hour", "location", "capacity", "occupancy_count"]
data = data.dropna(subset=required_columns).copy()

# Keep occupancy counts within each facility's capacity.
data["occupancy_count"] = data["occupancy_count"].clip(lower=0)
data["occupancy_count"] = np.minimum(data["occupancy_count"], data["capacity"])

# Recalculate percentage from count and capacity.
data["occupancy_pct"] = (data["occupancy_count"] / data["capacity"] * 100).round(1)

# Add human-readable and cyclical features.
day_names = {
    0: "Monday", 1: "Tuesday", 2: "Wednesday", 3: "Thursday",
    4: "Friday", 5: "Saturday", 6: "Sunday"
}
data["day_name"] = data["day_of_week"].map(day_names)
data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)
data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)
data["day_sin"] = np.sin(2 * np.pi * data["day_of_week"] / 7)
data["day_cos"] = np.cos(2 * np.pi * data["day_of_week"] / 7)

print(f"Rows after preprocessing: {len(data):,}")
display(data.head())

## 6. Check the cleaned dataset

Confirm that essential fields have no missing values and that occupancy values are valid after preprocessing.

In [ ]:
print("Missing values after preprocessing:")
display(data[required_columns + ["occupancy_pct"]].isna().sum().to_frame("missing_count"))

print("Occupancy count range:", data["occupancy_count"].min(), "to", data["occupancy_count"].max())
print("Occupancy percentage range:", data["occupancy_pct"].min(), "to", data["occupancy_pct"].max())
print("\nRecords by location:")
display(data["location"].value_counts().to_frame("record_count"))

## 7. Exploratory Data Analysis (EDA)

EDA helps us understand the distribution of the target variable and how crowd levels vary by location, time, and day.

### 7.1 Occupancy distribution

The histogram shows how frequently different occupancy counts occur.

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(data["occupancy_count"], bins=30, edgecolor="black")
plt.title("Distribution of Occupancy Count")
plt.xlabel("Number of people")
plt.ylabel("Frequency")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

### 7.2 Average occupancy by location

This comparison shows whether the two campus facilities have different typical crowd levels.

In [ ]:
avg_by_location = data.groupby("location", observed=False)["occupancy_count"].mean()

plt.figure(figsize=(7, 5))
plt.bar(avg_by_location.index.astype(str), avg_by_location.values, edgecolor="black")
plt.title("Average Occupancy by Location")
plt.xlabel("Location")
plt.ylabel("Average number of people")
plt.tight_layout()
plt.show()

display(avg_by_location.to_frame("average_occupancy").round(2))

### 7.3 Average occupancy by hour

This plot helps identify peak hours for the Library and Cafeteria.

In [ ]:
hourly = (
    data.groupby(["location", "hour"], observed=False)["occupancy_count"]
        .mean()
        .reset_index()
)

plt.figure(figsize=(11, 5))
for location in hourly["location"].astype(str).unique():
    subset = hourly[hourly["location"].astype(str) == location]
    plt.plot(subset["hour"], subset["occupancy_count"], marker="o", label=location)

plt.title("Average Occupancy by Hour and Location")
plt.xlabel("Hour of day")
plt.ylabel("Average number of people")
plt.xticks(sorted(data["hour"].unique()))
plt.legend(title="Location")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### 7.4 Average occupancy by day of the week

This analysis compares typical crowd levels across weekdays and weekends.

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
daily = (
    data.groupby(["day_name", "location"], observed=False)["occupancy_count"]
        .mean()
        .reset_index()
)
daily["day_name"] = pd.Categorical(daily["day_name"], categories=day_order, ordered=True)
daily = daily.sort_values("day_name")

x = np.arange(len(day_order))
width = 0.36
locations = ["Library", "Cafeteria"]

plt.figure(figsize=(12, 5))
for i, location in enumerate(locations):
    subset = daily[daily["location"].astype(str) == location].set_index("day_name")
    values = [subset.loc[day, "occupancy_count"] if day in subset.index else 0 for day in day_order]
    plt.bar(x + (i - 0.5) * width, values, width=width, label=location)

plt.title("Average Occupancy by Day of the Week")
plt.xlabel("Day")
plt.ylabel("Average number of people")
plt.xticks(x, day_order, rotation=20)
plt.legend()
plt.tight_layout()
plt.show()

### 7.5 Exam period and campus event analysis

Compare average occupancy when exam periods or campus events are flagged. These are synthetic indicators, so the chart reflects the assumptions used to generate this dataset.

In [ ]:
exam_summary = data.groupby("exam_period")["occupancy_count"].mean()
event_summary = data.groupby("campus_event")["occupancy_count"].mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(["No exam", "Exam period"], exam_summary.reindex([0, 1]).values, edgecolor="black")
axes[0].set_title("Average Occupancy: Exam Period")
axes[0].set_ylabel("Average number of people")

axes[1].bar(["No event", "Campus event"], event_summary.reindex([0, 1]).values, edgecolor="black")
axes[1].set_title("Average Occupancy: Campus Event")
axes[1].set_ylabel("Average number of people")

plt.tight_layout()
plt.show()

print("Exam-period means:")
display(exam_summary.to_frame("average_occupancy").round(2))
print("Campus-event means:")
display(event_summary.to_frame("average_occupancy").round(2))

### 7.6 Correlation analysis

Correlation measures the strength and direction of linear association between numerical variables. It does not prove causation. We exclude the date and encoded categorical location from this numeric correlation table.

In [ ]:
numeric_for_corr = [
    "day_of_week", "hour", "is_weekend", "exam_period",
    "campus_event", "capacity", "occupancy_count", "occupancy_pct"
]
corr = data[numeric_for_corr].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, aspect="auto", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
ax.set_title("Correlation Matrix")
fig.colorbar(im, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()

display(corr.round(2))

## 8. Key EDA summary

Run the cell below to produce concise statistics that can be used in the project report. Interpret the actual values printed by your notebook rather than copying assumed results.

In [ ]:
print("CAMPUSIQ — EDA SUMMARY")
print("=" * 45)
print(f"Total observations: {len(data):,}")
print(f"Date range: {data['date'].min().date()} to {data['date'].max().date()}")
print(f"Locations: {', '.join(map(str, data['location'].cat.categories))}")
print(f"Average occupancy: {data['occupancy_count'].mean():.2f} people")
print(f"Median occupancy: {data['occupancy_count'].median():.2f} people")
print(f"Minimum occupancy: {data['occupancy_count'].min()} people")
print(f"Maximum occupancy: {data['occupancy_count'].max()} people")

print("\nAverage occupancy by location:")
print(data.groupby("location", observed=False)["occupancy_count"].mean().round(2))

print("\nPeak average hour by location:")
print(hourly.loc[hourly.groupby("location")["occupancy_count"].idxmax()].to_string(index=False))

## 9. Save the preprocessed dataset

The cleaned dataset can be used in the next stage: feature selection, train/test splitting, and regression model training.

In [ ]:
OUTPUT_PATH = Path("data") / "campus_crowd_preprocessed.csv"
data.to_csv(OUTPUT_PATH, index=False)
print(f"Preprocessed dataset saved to: {OUTPUT_PATH}")

## 10. Conclusion

In this notebook, we:
- Loaded and inspected the synthetic campus occupancy dataset.
- Checked missing values, duplicates, and invalid occupancy values.
- Prepared numeric and categorical fields.
- Created day-name and cyclical time features.
- Explored occupancy distributions and patterns by location, hour, weekday, exams, and campus events.
- Saved the cleaned dataset for machine learning.

**Next stage:** Use the preprocessed data to train and compare Linear Regression, Decision Tree, Random Forest, and KNN regression models using a chronological train/test split.